
# 03N — Freeze Corrected Wildfire Fireline ML Cohort

**Notebook build ID: `03N_FREEZE_CORRECTED_ML_COHORT_V1`**

03N finalizes the corrected target dataset after the 03J–03M representation and evidence audits.

No NIROPS rasterization is performed.

No geometry thresholds are reconsidered.

No target threshold search is performed.

---

# Frozen methodology

## Corrected local fireline unit

\[
L_{\max}=250\text{ m}
\]

with local short-fragment connection:

\[
g_{\max}=10\text{ m}.
\]

Within-unit NIROPS evidence was sampled at approximately:

\[
25\text{ m}
\]

with at least one sample per constituent geometry part.

---

# Arc-length sample weighting

For geometry part \(j\) of length \(L_j\) represented by \(n_j\) samples:

\[
w_{jp}=\frac{L_j}{n_j}.
\]

Weighted support and conflict coverage are:

\[
f_S=
\frac{\sum_p w_p\,\mathbf{1}[p\text{ supported}]}
{L_{\mathrm{unit}}},
\]

\[
f_C=
\frac{\sum_p w_p\,\mathbf{1}[p\text{ conflict}]}
{L_{\mathrm{unit}}}.
\]

---

# Final target rule

## Burned Over

A corrected unit is temporally corroborated when:

\[
\boxed{f_S\ge0.10}
\]

## Held

A corrected unit is temporally corroborated when:

\[
\boxed{
f_S\ge0.10
\quad\land\quad
f_C=0
}
\]

The 10% threshold requires approximately one normal 25 m-scale portion of a full 250 m unit to contain supported temporal evidence.

The zero-conflict Held rule preserves the interpretation that no sampled portion of the corrected local line shows contradictory penetration evidence.

---

# Final binary target

Among temporally corroborated units:

\[
y=
\begin{cases}
1, & \text{Held}\\
0, & \text{Burned Over}
\end{cases}
\]

The column is stored as:

```text
hold_label
```

---

# Final challenge interval

For every temporally corroborated unit, supported sample-point events are grouped by NIROPS interval.

For interval \(I\):

\[
W_I=
\sum_{p\in I}w_p.
\]

The primary challenge interval is:

\[
\boxed{
I^*
=
\arg\max_I W_I
}
\]

with deterministic tie-breaking toward the earlier interval.

Timing confidence is:

\[
C=
\frac{W_{I^*}}
{\sum_IW_I}.
\]

A unit has a well-defined primary dynamic interval when:

\[
\boxed{C\ge0.50}.
\]

This is stored as:

```text
challenge_timing_confident
```

Units below 0.50 remain valid outcome-labeled units but should not be used in analyses requiring one unique dynamic weather interval.

---

# Timestamp correction

03M revealed that cached integer timestamps can be stored at different datetime resolutions depending on the Pandas datetime dtype.

03N does **not** assume nanoseconds.

For each trajectory cache, the timestamp unit is inferred from integer magnitude:

- nanoseconds: approximately \(10^{18}\),
- microseconds: approximately \(10^{15}\),
- milliseconds: approximately \(10^{12}\),
- seconds: approximately \(10^9\).

The inferred timestamps are then validated against plausible positive NIROPS interval durations.

The final frozen cohort therefore stores corrected UTC challenge timestamps.

---

# Outputs

```text
data\processed\fireline_corrected_local_250m_10m\frozen_cohort\
    fireline_ml_cohort_frozen_v1.parquet
    fireline_ml_dynamic_timing_cohort_frozen_v1.parquet
    fireline_ml_excluded_units_v1.parquet
    DATASET_CARD_FROZEN_V1.json

outputs\frozen_corrected_ml_cohort\
    FREEZE_SUMMARY.csv
    CLASS_SUMMARY.csv
    YEAR_CLASS_SUMMARY.csv
    INCIDENT_CONCENTRATION.csv
    TIMING_CONFIDENCE_SUMMARY.csv
    EXCLUSION_SUMMARY.csv
    FROZEN_COHORT_REPORT.md
```


In [ ]:
import os  # Configurable project root for the public release.

NOTEBOOK_BUILD_ID = "03N_FREEZE_CORRECTED_ML_COHORT_V1"

from pathlib import Path
import sys
import subprocess
import importlib.util

PROJECT_ROOT = Path(
    os.environ.get("FIRELINE_PROJECT_ROOT", ".")
)

PROCESSED_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "fireline_corrected_local_250m_10m"
)

UNIT_ROOT = (
    PROCESSED_ROOT
    / "units"
)

POINT_EVIDENCE_ROOT = (
    PROCESSED_ROOT
    / "point_evidence"
)

TRAJECTORY_ROOT = (
    PROCESSED_ROOT
    / "trajectories"
)

WEIGHTED_METRICS_PATH = (
    PROCESSED_ROOT
    / "unit_evidence_rule_audit"
    / "weighted_unit_evidence_metrics.parquet"
)

FROZEN_ROOT = (
    PROCESSED_ROOT
    / "frozen_cohort"
)

TIMING_SHARD_ROOT = (
    FROZEN_ROOT
    / "timing_shards"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / "frozen_corrected_ml_cohort"
)

for path in [
    FROZEN_ROOT,
    TIMING_SHARD_ROOT,
    OUTPUT_ROOT,
]:

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

assert UNIT_ROOT.exists(), UNIT_ROOT
assert POINT_EVIDENCE_ROOT.exists(), POINT_EVIDENCE_ROOT
assert TRAJECTORY_ROOT.exists(), TRAJECTORY_ROOT
assert WEIGHTED_METRICS_PATH.exists(), WEIGHTED_METRICS_PATH

# ============================================================
# FROZEN TARGET SETTINGS
# ============================================================

SUPPORT_FRACTION_THRESHOLD = 0.10

HELD_CONFLICT_FRACTION_MAX = 0.0

CHALLENGE_TIMING_CONFIDENCE_MIN = 0.50

EXPECTED_MAX_UNIT_LENGTH_M = 250.0
EXPECTED_FRAGMENT_GAP_M = 10.0
EXPECTED_SAMPLE_SPACING_M = 25.0
EXPECTED_CHALLENGE_TOLERANCE_M = 50.0
EXPECTED_MAX_NIROPS_GAP_HOURS = 48.0

RESUME = True

print("=" * 84)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("=" * 84)

print(
    "Final support threshold:",
    SUPPORT_FRACTION_THRESHOLD,
)

print(
    "Final Held conflict tolerance:",
    HELD_CONFLICT_FRACTION_MAX,
)

print(
    "Dynamic timing-confidence threshold:",
    CHALLENGE_TIMING_CONFIDENCE_MIN,
)


## 1. Imports

In [ ]:

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "geopandas": "geopandas",
    "pyarrow": "pyarrow",
    "tqdm": "tqdm",
}

for module, package in required.items():

    if importlib.util.find_spec(
        module
    ) is None:

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
            ]
        )

import json
import gc
import time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd

from tqdm.auto import tqdm

pd.set_option(
    "display.max_columns",
    300,
)

pd.set_option(
    "display.max_rows",
    300,
)

pd.set_option(
    "display.width",
    220,
)

print(
    "Python:",
    sys.executable,
)

print(
    "Pandas:",
    pd.__version__,
)

print(
    "GeoPandas:",
    gpd.__version__,
)


## 2. Load 03M weighted unit evidence metrics

In [ ]:

unit_metrics = pd.read_parquet(
    WEIGHTED_METRICS_PATH
)

assert len(
    unit_metrics
) > 0

assert unit_metrics[
    "unit_uid"
].is_unique

required_columns = [
    "unit_uid",
    "unit_row_index",
    "_irwin_norm",
    "IncidentName",
    "Year",
    "FeatureCategory",
    "_engagement_norm",
    "_geom_hash",
    "source_index",
    "unit_origin",
    "actual_line_length_m",
    "constituent_components",
    "max_gap_used_m",
    "sample_points",
    "weighted_support_fraction",
    "weighted_conflict_fraction",
    "weighted_insufficient_fraction",
]

missing_columns = [
    column
    for column in required_columns
    if column
    not in unit_metrics.columns
]

if missing_columns:

    raise RuntimeError(
        "03M weighted metrics missing required columns: "
        + str(
            missing_columns
        )
    )

print(
    "Corrected units:",
    f"{len(unit_metrics):,}"
)

print(
    "Incidents:",
    unit_metrics[
        "_irwin_norm"
    ].nunique(),
)

print(
    "Labels:",
    unit_metrics[
        "_engagement_norm"
    ].value_counts(
        dropna=False
    ).to_dict(),
)


## 3. Apply the frozen target rule

In [ ]:

is_held = (
    unit_metrics[
        "_engagement_norm"
    ]
    == "Held"
)

is_burned_over = (
    unit_metrics[
        "_engagement_norm"
    ]
    == "Burned Over"
)

support_ok = (
    unit_metrics[
        "weighted_support_fraction"
    ]
    >= (
        SUPPORT_FRACTION_THRESHOLD
        - 1e-12
    )
)

held_conflict_ok = (
    unit_metrics[
        "weighted_conflict_fraction"
    ]
    <= (
        HELD_CONFLICT_FRACTION_MAX
        + 1e-12
    )
)

held_supported = (
    is_held
    & support_ok
    & held_conflict_ok
)

burned_over_supported = (
    is_burned_over
    & support_ok
)

final_supported_mask = (
    held_supported
    | burned_over_supported
)

unit_metrics[
    "final_temporally_corroborated"
] = (
    final_supported_mask
)

unit_metrics[
    "final_exclusion_reason"
] = "included"

unit_metrics.loc[
    is_held
    & ~support_ok,
    "final_exclusion_reason",
] = "held_support_fraction_below_0.10"

unit_metrics.loc[
    is_held
    & support_ok
    & ~held_conflict_ok,
    "final_exclusion_reason",
] = "held_nonzero_conflict_fraction"

unit_metrics.loc[
    is_burned_over
    & ~support_ok,
    "final_exclusion_reason",
] = "burned_over_support_fraction_below_0.10"

other_label = (
    ~is_held
    & ~is_burned_over
)

unit_metrics.loc[
    other_label,
    "final_exclusion_reason",
] = "unexpected_engagement_label"

frozen_base = (
    unit_metrics[
        final_supported_mask
    ]
    .copy()
)

frozen_base[
    "hold_label"
] = (
    frozen_base[
        "_engagement_norm"
    ]
    == "Held"
).astype(
    np.int8
)

excluded_units = (
    unit_metrics[
        ~final_supported_mask
    ]
    .copy()
)

print(
    "Frozen target candidates:",
    f"{len(frozen_base):,}"
)

print(
    "Held:",
    f"{(frozen_base['_engagement_norm'] == 'Held').sum():,}"
)

print(
    "Burned Over:",
    f"{(frozen_base['_engagement_norm'] == 'Burned Over').sum():,}"
)

print(
    "Excluded:",
    f"{len(excluded_units):,}"
)



# 4. Robust cached timestamp decoder

The trajectory cache stores raw integer datetime values from the Pandas datetime index.

03N infers the physical unit from integer magnitude rather than assuming nanoseconds.

The decoder also validates that:

1. timestamps are strictly increasing;
2. positive interval durations are finite;
3. the median interval is not implausibly tiny or enormous.

The unit selected by magnitude is saved for audit.


In [ ]:

def infer_integer_datetime_unit(
    raw_values,
):

    raw_values = np.asarray(
        raw_values,
        dtype=np.int64,
    )

    finite_values = raw_values[
        raw_values
        != np.iinfo(
            np.int64
        ).min
    ]

    if not len(
        finite_values
    ):

        raise RuntimeError(
            "Trajectory timestamp cache contains no valid integers."
        )

    magnitude = float(
        np.median(
            np.abs(
                finite_values.astype(
                    np.float64
                )
            )
        )
    )

    if magnitude >= 1e17:

        return "ns"

    if magnitude >= 1e14:

        return "us"

    if magnitude >= 1e11:

        return "ms"

    return "s"


def decode_cached_times(
    raw_values,
):

    raw_values = np.asarray(
        raw_values,
        dtype=np.int64,
    )

    unit = infer_integer_datetime_unit(
        raw_values
    )

    times = pd.DatetimeIndex(
        pd.to_datetime(
            raw_values,
            unit=unit,
            utc=True,
        )
    )

    if not times.is_monotonic_increasing:

        raise RuntimeError(
            "Decoded cached NIROPS timestamps are not monotonically increasing."
        )

    if len(
        times
    ) >= 2:

        gap_series = (
            pd.Series(
                times
            )
            .diff()
            .dt.total_seconds()
            .dropna()
            / 3600.0
        )

        positive = gap_series[
            gap_series
            > 0
        ].to_numpy(
            dtype=float
        )

        if not len(
            positive
        ):

            raise RuntimeError(
                "Decoded NIROPS timestamps contain no positive intervals."
            )

        median_gap_hours = float(
            np.median(
                positive
            )
        )

        max_gap_hours = float(
            np.max(
                positive
            )
        )

        # Broad sanity bounds. This validates unit decoding without
        # requiring every operational NIROPS gap to be <=48 h.
        if (
            median_gap_hours
            < 0.25
            or median_gap_hours
            > 240.0
        ):

            raise RuntimeError(
                "Decoded timestamp scale is implausible: "
                f"unit={unit}, median gap={median_gap_hours:.6f} h"
            )

    else:

        median_gap_hours = np.nan
        max_gap_hours = np.nan

    return (
        times,
        unit,
        median_gap_hours,
        max_gap_hours,
    )



# 5. Reconstruct part-length weights for one incident

This repeats only the cheap geometry-part weighting operation from 03M.

No NIROPS geometry is loaded.

No raster is created.


In [ ]:

def build_part_length_table(
    units,
):

    exploded = (
        units[
            [
                "unit_row_index",
                "geometry",
            ]
        ]
        .set_index(
            "unit_row_index"
        )
        .explode(
            index_parts=True
        )
        .reset_index()
    )

    if (
        "level_1"
        in exploded.columns
    ):

        exploded = exploded.rename(
            columns={
                "level_1": "part_index",
            }
        )

    elif (
        "part_index"
        not in exploded.columns
    ):

        candidates = [
            column
            for column in exploded.columns
            if column
            not in {
                "unit_row_index",
                "geometry",
            }
        ]

        if len(
            candidates
        ) != 1:

            raise RuntimeError(
                "Could not identify exploded part index."
            )

        exploded = exploded.rename(
            columns={
                candidates[
                    0
                ]: "part_index",
            }
        )

    exploded[
        "part_index"
    ] = (
        exploded[
            "part_index"
        ]
        .astype(
            np.int16
        )
    )

    exploded[
        "part_length_m"
    ] = (
        exploded.geometry.length
        .astype(
            float
        )
    )

    return exploded[
        [
            "unit_row_index",
            "part_index",
            "part_length_m",
        ]
    ]



# 6. Dominant supported interval for one incident

Only units that survive the final target rule are needed.

For each supported sample-point event:

\[
w_p=L_{\mathrm{part}}/n_{\mathrm{samples,part}}.
\]

Weights are summed within each event interval, and the interval with maximum supported line weight is selected.

Tie-breaking:

1. larger supported arc-length weight;
2. earlier interval end;
3. earlier interval start.

This makes the result deterministic.


In [ ]:

def build_incident_final_timing(
    irwin,
    included_unit_uids,
):

    unit_path = (
        UNIT_ROOT
        / f"{irwin}.parquet"
    )

    point_path = (
        POINT_EVIDENCE_ROOT
        / f"{irwin}.parquet"
    )

    trajectory_path = (
        TRAJECTORY_ROOT
        / f"{irwin}.npz"
    )

    assert unit_path.exists(), unit_path
    assert point_path.exists(), point_path
    assert trajectory_path.exists(), trajectory_path

    units = gpd.read_parquet(
        unit_path
    )

    units = units[
        units[
            "unit_uid"
        ].isin(
            included_unit_uids
        )
    ].copy()

    if not len(
        units
    ):

        return (
            pd.DataFrame(),
            {
                "_irwin_norm": irwin,
                "included_units": 0,
            },
        )

    included_indices = set(
        units[
            "unit_row_index"
        ].astype(
            int
        )
    )

    points = pd.read_parquet(
        point_path
    )

    points = points[
        points[
            "unit_row_index"
        ].isin(
            included_indices
        )
    ].copy()

    with np.load(
        trajectory_path,
        allow_pickle=False,
    ) as cache:

        raw_times = (
            cache[
                "times_ns"
            ]
            .copy()
        )

    (
        times,
        timestamp_unit,
        median_gap_hours,
        max_gap_hours,
    ) = decode_cached_times(
        raw_times
    )

    part_lengths = build_part_length_table(
        units
    )

    samples_per_part = (
        points.groupby(
            [
                "unit_row_index",
                "part_index",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size": "samples_in_part",
            }
        )
    )

    points = (
        points.merge(
            part_lengths,
            on=[
                "unit_row_index",
                "part_index",
            ],
            how="left",
            validate="many_to_one",
        )
        .merge(
            samples_per_part,
            on=[
                "unit_row_index",
                "part_index",
            ],
            how="left",
            validate="many_to_one",
        )
    )

    if points[
        "part_length_m"
    ].isna().any():

        raise RuntimeError(
            f"Missing part lengths for incident {irwin}."
        )

    points[
        "sample_length_weight_m"
    ] = (
        points[
            "part_length_m"
        ]
        / points[
            "samples_in_part"
        ]
    )

    supported_events = points[
        (
            points[
                "point_evidence"
            ]
            == "temporally_supported"
        )
        & (
            points[
                "event_start_idx"
            ]
            >= 0
        )
        & (
            points[
                "event_end_idx"
            ]
            >= 0
        )
    ][
        [
            "unit_row_index",
            "unit_uid",
            "event_start_idx",
            "event_end_idx",
            "sample_length_weight_m",
        ]
    ].copy()

    if not len(
        supported_events
    ):

        result = pd.DataFrame(
            {
                "unit_uid": units[
                    "unit_uid"
                ],
            }
        )

        for column in [
            "dominant_supported_start_idx",
            "dominant_supported_end_idx",
            "supported_interval_count",
        ]:

            result[
                column
            ] = pd.NA

        for column in [
            "dominant_supported_weight_m",
            "total_supported_event_weight_m",
            "dominant_interval_weight_fraction",
            "supported_event_span_hours",
            "dominant_challenge_interval_hours",
        ]:

            result[
                column
            ] = np.nan

        for column in [
            "dominant_challenge_interval_start_utc",
            "dominant_challenge_interval_end_utc",
            "earliest_supported_interval_start_utc",
            "latest_supported_interval_end_utc",
        ]:

            result[
                column
            ] = pd.NaT

    else:

        interval_weights = (
            supported_events.groupby(
                [
                    "unit_row_index",
                    "unit_uid",
                    "event_start_idx",
                    "event_end_idx",
                ],
                as_index=False,
            )
            .agg(
                interval_supported_weight_m=(
                    "sample_length_weight_m",
                    "sum",
                ),
                interval_supported_points=(
                    "sample_length_weight_m",
                    "size",
                ),
            )
        )

        interval_stats = (
            interval_weights.groupby(
                [
                    "unit_row_index",
                    "unit_uid",
                ],
                as_index=False,
            )
            .agg(
                supported_interval_count=(
                    "event_end_idx",
                    "size",
                ),
                earliest_supported_start_idx=(
                    "event_start_idx",
                    "min",
                ),
                latest_supported_end_idx=(
                    "event_end_idx",
                    "max",
                ),
                total_supported_event_weight_m=(
                    "interval_supported_weight_m",
                    "sum",
                ),
            )
        )

        dominant = (
            interval_weights.sort_values(
                [
                    "unit_row_index",
                    "interval_supported_weight_m",
                    "event_end_idx",
                    "event_start_idx",
                ],
                ascending=[
                    True,
                    False,
                    True,
                    True,
                ],
            )
            .drop_duplicates(
                "unit_row_index",
                keep="first",
            )
            .rename(
                columns={
                    "event_start_idx": "dominant_supported_start_idx",
                    "event_end_idx": "dominant_supported_end_idx",
                    "interval_supported_weight_m": "dominant_supported_weight_m",
                }
            )
        )

        result = (
            interval_stats.merge(
                dominant[
                    [
                        "unit_row_index",
                        "unit_uid",
                        "dominant_supported_start_idx",
                        "dominant_supported_end_idx",
                        "dominant_supported_weight_m",
                    ]
                ],
                on=[
                    "unit_row_index",
                    "unit_uid",
                ],
                how="left",
                validate="one_to_one",
            )
        )

        result[
            "dominant_interval_weight_fraction"
        ] = (
            result[
                "dominant_supported_weight_m"
            ]
            / result[
                "total_supported_event_weight_m"
            ]
        )

        def timestamp_at(
            value,
        ):

            return times[
                int(
                    value
                )
            ]

        result[
            "dominant_challenge_interval_start_utc"
        ] = [
            timestamp_at(
                value
            )
            for value in result[
                "dominant_supported_start_idx"
            ]
        ]

        result[
            "dominant_challenge_interval_end_utc"
        ] = [
            timestamp_at(
                value
            )
            for value in result[
                "dominant_supported_end_idx"
            ]
        ]

        result[
            "earliest_supported_interval_start_utc"
        ] = [
            timestamp_at(
                value
            )
            for value in result[
                "earliest_supported_start_idx"
            ]
        ]

        result[
            "latest_supported_interval_end_utc"
        ] = [
            timestamp_at(
                value
            )
            for value in result[
                "latest_supported_end_idx"
            ]
        ]

        result[
            "dominant_challenge_interval_hours"
        ] = (
            pd.to_datetime(
                result[
                    "dominant_challenge_interval_end_utc"
                ],
                utc=True,
            )
            - pd.to_datetime(
                result[
                    "dominant_challenge_interval_start_utc"
                ],
                utc=True,
            )
        ).dt.total_seconds() / 3600.0

        result[
            "supported_event_span_hours"
        ] = (
            pd.to_datetime(
                result[
                    "latest_supported_interval_end_utc"
                ],
                utc=True,
            )
            - pd.to_datetime(
                result[
                    "earliest_supported_interval_start_utc"
                ],
                utc=True,
            )
        ).dt.total_seconds() / 3600.0

    # Ensure every included unit appears, even if an unexpected unit
    # has no supported point-level event after final filtering.
    result = (
        pd.DataFrame(
            {
                "unit_uid": units[
                    "unit_uid"
                ].to_numpy(),
            }
        )
        .merge(
            result,
            on="unit_uid",
            how="left",
            validate="one_to_one",
        )
    )

    result[
        "_irwin_norm"
    ] = str(
        irwin
    )

    result[
        "timestamp_integer_unit"
    ] = timestamp_unit

    result[
        "incident_median_nirops_gap_hours"
    ] = median_gap_hours

    result[
        "incident_max_nirops_gap_hours"
    ] = max_gap_hours

    metadata = {
        "_irwin_norm": irwin,
        "included_units": len(
            units
        ),
        "units_with_supported_interval": int(
            result[
                "dominant_challenge_interval_start_utc"
            ].notna().sum()
        ),
        "timestamp_integer_unit": timestamp_unit,
        "median_nirops_gap_hours": median_gap_hours,
        "max_nirops_gap_hours": max_gap_hours,
    }

    return (
        result,
        metadata,
    )


## 7. Reconstruct final challenge timing incident-by-incident

In [ ]:

TIMING_RUN_LOG_PATH = (
    OUTPUT_ROOT
    / "TIMING_RECONSTRUCTION_RUN_LOG.csv"
)

incident_groups = (
    frozen_base.groupby(
        "_irwin_norm",
        sort=False,
    )
)

timing_run_rows = []

outer = tqdm(
    incident_groups,
    total=incident_groups.ngroups,
    desc="03N final timing incidents",
)

for (
    irwin,
    group,
) in outer:

    irwin = str(
        irwin
    )

    output_path = (
        TIMING_SHARD_ROOT
        / f"{irwin}.parquet"
    )

    start = time.perf_counter()

    try:

        if (
            RESUME
            and output_path.exists()
        ):

            preview = pd.read_parquet(
                output_path,
                columns=[
                    "unit_uid",
                ],
            )

            timing_run_rows.append(
                {
                    "_irwin_norm": irwin,
                    "status": "SKIPPED_COMPLETE",
                    "included_units": len(
                        preview
                    ),
                    "runtime_seconds": (
                        time.perf_counter()
                        - start
                    ),
                }
            )

            outer.set_postfix(
                {
                    "incident": irwin[
                        :12
                    ],
                    "status": "resume",
                }
            )

            continue

        (
            timing,
            metadata,
        ) = build_incident_final_timing(
            irwin=irwin,
            included_unit_uids=set(
                group[
                    "unit_uid"
                ]
            ),
        )

        timing.to_parquet(
            output_path,
            index=False,
        )

        metadata[
            "status"
        ] = "OK"

        metadata[
            "runtime_seconds"
        ] = (
            time.perf_counter()
            - start
        )

        timing_run_rows.append(
            metadata
        )

        outer.set_postfix(
            {
                "incident": irwin[
                    :12
                ],
                "units": metadata[
                    "included_units"
                ],
                "unit": metadata[
                    "timestamp_integer_unit"
                ],
                "sec": f"{metadata['runtime_seconds']:.2f}",
            }
        )

        del timing
        gc.collect()

    except Exception as exc:

        timing_run_rows.append(
            {
                "_irwin_norm": irwin,
                "status": "ERROR",
                "error": (
                    f"{type(exc).__name__}: {exc}"
                ),
                "runtime_seconds": (
                    time.perf_counter()
                    - start
                ),
            }
        )

        outer.set_postfix(
            {
                "incident": irwin[
                    :12
                ],
                "status": "ERROR",
            }
        )

    pd.DataFrame(
        timing_run_rows
    ).to_csv(
        TIMING_RUN_LOG_PATH,
        index=False,
    )


## 8. Combine final timing shards

In [ ]:

timing_files = sorted(
    TIMING_SHARD_ROOT.glob(
        "*.parquet"
    )
)

expected_timing_incidents = (
    frozen_base[
        "_irwin_norm"
    ].nunique()
)

print(
    "Timing shards:",
    len(
        timing_files
    ),
    "/",
    expected_timing_incidents,
)

if (
    len(
        timing_files
    )
    != expected_timing_incidents
):

    raise RuntimeError(
        "Final timing reconstruction is incomplete. "
        "Review TIMING_RECONSTRUCTION_RUN_LOG.csv and rerun."
    )

timing_frames = []

for path in tqdm(
    timing_files,
    desc="Combine final timing",
):

    timing_frames.append(
        pd.read_parquet(
            path
        )
    )

final_timing = pd.concat(
    timing_frames,
    ignore_index=True,
)

del timing_frames

assert final_timing[
    "unit_uid"
].is_unique

print(
    "Final timing rows:",
    f"{len(final_timing):,}"
)

if (
    len(
        final_timing
    )
    != len(
        frozen_base
    )
):

    raise RuntimeError(
        "Timing rows do not match frozen target candidates."
    )


## 9. Build the final frozen cohort

In [ ]:

frozen = (
    frozen_base.merge(
        final_timing,
        on="unit_uid",
        how="left",
        validate="one_to_one",
        suffixes=(
            "",
            "_timing",
        ),
    )
)

frozen[
    "challenge_timing_confident"
] = (
    frozen[
        "dominant_interval_weight_fraction"
    ]
    >= (
        CHALLENGE_TIMING_CONFIDENCE_MIN
        - 1e-12
    )
)

frozen[
    "has_supported_challenge_interval"
] = (
    frozen[
        "dominant_challenge_interval_start_utc"
    ].notna()
    & frozen[
        "dominant_challenge_interval_end_utc"
    ].notna()
)

frozen[
    "dynamic_timing_eligible"
] = (
    frozen[
        "has_supported_challenge_interval"
    ]
    & frozen[
        "challenge_timing_confident"
    ]
)

# Scientific rule provenance.
frozen[
    "target_rule_version"
] = (
    "FROZEN_V1"
)

frozen[
    "support_fraction_threshold"
] = (
    SUPPORT_FRACTION_THRESHOLD
)

frozen[
    "held_conflict_fraction_max"
] = (
    HELD_CONFLICT_FRACTION_MAX
)

frozen[
    "timing_confidence_threshold"
] = (
    CHALLENGE_TIMING_CONFIDENCE_MIN
)

# Sort deterministically.
frozen = (
    frozen.sort_values(
        [
            "Year",
            "_irwin_norm",
            "_geom_hash",
            "unit_row_index",
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)

frozen[
    "frozen_row_index"
] = np.arange(
    len(
        frozen
    ),
    dtype=np.int32,
)

if not frozen[
    "unit_uid"
].is_unique:

    raise RuntimeError(
        "Duplicate unit_uid in final frozen cohort."
    )

if not (
    frozen[
        "weighted_support_fraction"
    ]
    >= (
        SUPPORT_FRACTION_THRESHOLD
        - 1e-12
    )
).all():

    raise RuntimeError(
        "Final cohort contains a unit below the frozen support threshold."
    )

held_final = (
    frozen[
        "_engagement_norm"
    ]
    == "Held"
)

if not (
    frozen.loc[
        held_final,
        "weighted_conflict_fraction",
    ]
    <= (
        HELD_CONFLICT_FRACTION_MAX
        + 1e-12
    )
).all():

    raise RuntimeError(
        "Final Held cohort contains nonzero conflict coverage."
    )

print(
    "FINAL FROZEN COHORT:",
    f"{len(frozen):,}"
)

print(
    "Incidents:",
    frozen[
        "_irwin_norm"
    ].nunique(),
)

print(
    "Held:",
    f"{(frozen['hold_label'] == 1).sum():,}"
)

print(
    "Burned Over:",
    f"{(frozen['hold_label'] == 0).sum():,}"
)

print(
    "Dynamic timing eligible:",
    f"{frozen['dynamic_timing_eligible'].sum():,}",
)


## 10. Save frozen cohorts

In [ ]:

FROZEN_COHORT_PATH = (
    FROZEN_ROOT
    / "fireline_ml_cohort_frozen_v1.parquet"
)

DYNAMIC_COHORT_PATH = (
    FROZEN_ROOT
    / "fireline_ml_dynamic_timing_cohort_frozen_v1.parquet"
)

EXCLUDED_PATH = (
    FROZEN_ROOT
    / "fireline_ml_excluded_units_v1.parquet"
)

frozen.to_parquet(
    FROZEN_COHORT_PATH,
    index=False,
)

dynamic_frozen = (
    frozen[
        frozen[
            "dynamic_timing_eligible"
        ]
    ]
    .copy()
)

dynamic_frozen.to_parquet(
    DYNAMIC_COHORT_PATH,
    index=False,
)

excluded_units.to_parquet(
    EXCLUDED_PATH,
    index=False,
)

print(
    "Frozen cohort:",
    FROZEN_COHORT_PATH
)

print(
    "Dynamic timing cohort:",
    DYNAMIC_COHORT_PATH
)

print(
    "Excluded units:",
    EXCLUDED_PATH
)


## 11. Freeze summary

In [ ]:

freeze_summary = pd.DataFrame(
    [
        {
            "source_corrected_units": len(
                unit_metrics
            ),
            "frozen_supported_units": len(
                frozen
            ),
            "excluded_units": len(
                excluded_units
            ),
            "frozen_incidents": frozen[
                "_irwin_norm"
            ].nunique(),
            "held_units": int(
                (
                    frozen[
                        "hold_label"
                    ]
                    == 1
                ).sum()
            ),
            "burned_over_units": int(
                (
                    frozen[
                        "hold_label"
                    ]
                    == 0
                ).sum()
            ),
            "dynamic_timing_eligible_units": int(
                frozen[
                    "dynamic_timing_eligible"
                ].sum()
            ),
            "dynamic_timing_eligible_pct": (
                frozen[
                    "dynamic_timing_eligible"
                ].mean()
                * 100.0
            ),
            "timing_ambiguous_or_missing_units": int(
                (
                    ~frozen[
                        "dynamic_timing_eligible"
                    ]
                ).sum()
            ),
            "support_threshold": SUPPORT_FRACTION_THRESHOLD,
            "held_conflict_tolerance": HELD_CONFLICT_FRACTION_MAX,
            "timing_confidence_threshold": CHALLENGE_TIMING_CONFIDENCE_MIN,
        }
    ]
)

display(
    freeze_summary
)

freeze_summary.to_csv(
    OUTPUT_ROOT
    / "FREEZE_SUMMARY.csv",
    index=False,
)


## 12. Class summary

In [ ]:

class_summary = (
    frozen.groupby(
        "_engagement_norm",
        dropna=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        parent_geometries=(
            "_geom_hash",
            "nunique",
        ),
        total_line_length_km=(
            "actual_line_length_m",
            lambda s: (
                s.sum()
                / 1000.0
            ),
        ),
        median_unit_length_m=(
            "actual_line_length_m",
            "median",
        ),
        median_weighted_support_fraction=(
            "weighted_support_fraction",
            "median",
        ),
        dynamic_timing_eligible=(
            "dynamic_timing_eligible",
            "sum",
        ),
    )
    .reset_index()
)

class_summary[
    "class_pct"
] = (
    class_summary[
        "units"
    ]
    / class_summary[
        "units"
    ].sum()
    * 100.0
)

class_summary[
    "dynamic_timing_eligible_pct"
] = (
    class_summary[
        "dynamic_timing_eligible"
    ]
    / class_summary[
        "units"
    ]
    * 100.0
)

display(
    class_summary
)

class_summary.to_csv(
    OUTPUT_ROOT
    / "CLASS_SUMMARY.csv",
    index=False,
)


## 13. Year × class summary

In [ ]:

year_class_summary = (
    frozen.groupby(
        [
            "Year",
            "_engagement_norm",
        ],
        dropna=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        dynamic_timing_eligible=(
            "dynamic_timing_eligible",
            "sum",
        ),
    )
    .reset_index()
)

display(
    year_class_summary
)

year_class_summary.to_csv(
    OUTPUT_ROOT
    / "YEAR_CLASS_SUMMARY.csv",
    index=False,
)


## 14. Incident representation

In [ ]:

incident_class = (
    frozen.groupby(
        [
            "_irwin_norm",
            "_engagement_norm",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

for column in [
    "Held",
    "Burned Over",
]:

    if column not in (
        incident_class.columns
    ):

        incident_class[
            column
        ] = 0

incident_representation = pd.DataFrame(
    [
        {
            "frozen_incidents": len(
                incident_class
            ),
            "incidents_with_held": int(
                (
                    incident_class[
                        "Held"
                    ]
                    > 0
                ).sum()
            ),
            "incidents_with_burned_over": int(
                (
                    incident_class[
                        "Burned Over"
                    ]
                    > 0
                ).sum()
            ),
            "incidents_with_both_classes": int(
                (
                    (
                        incident_class[
                            "Held"
                        ]
                        > 0
                    )
                    & (
                        incident_class[
                            "Burned Over"
                        ]
                        > 0
                    )
                ).sum()
            ),
            "incidents_with_at_least_10_each": int(
                (
                    (
                        incident_class[
                            "Held"
                        ]
                        >= 10
                    )
                    & (
                        incident_class[
                            "Burned Over"
                        ]
                        >= 10
                    )
                ).sum()
            ),
        }
    ]
)

display(
    incident_representation
)

incident_representation.to_csv(
    OUTPUT_ROOT
    / "INCIDENT_REPRESENTATION.csv",
    index=False,
)


## 15. Incident concentration

In [ ]:

def concentration_metrics(
    frame,
    label,
):

    counts = (
        frame.groupby(
            "_irwin_norm"
        )
        .size()
        .sort_values(
            ascending=False
        )
        .to_numpy(
            dtype=float
        )
    )

    if not len(
        counts
    ):

        return {
            "group": label,
        }

    total = counts.sum()

    def top_pct(
        n,
    ):

        return (
            counts[
                :min(
                    n,
                    len(
                        counts
                    )
                )
            ].sum()
            / total
            * 100.0
        )

    return {
        "group": label,
        "incidents": len(
            counts
        ),
        "units": int(
            total
        ),
        "top1_pct": top_pct(
            1
        ),
        "top5_pct": top_pct(
            5
        ),
        "top10_pct": top_pct(
            10
        ),
        "top20_pct": top_pct(
            20
        ),
        "effective_incidents": (
            total**2
            / np.square(
                counts
            ).sum()
        ),
    }


concentration_rows = [
    concentration_metrics(
        frozen,
        "All frozen",
    )
]

for label in [
    "Held",
    "Burned Over",
]:

    concentration_rows.append(
        concentration_metrics(
            frozen[
                frozen[
                    "_engagement_norm"
                ]
                == label
            ],
            label,
        )
    )

incident_concentration = pd.DataFrame(
    concentration_rows
)

display(
    incident_concentration
)

incident_concentration.to_csv(
    OUTPUT_ROOT
    / "INCIDENT_CONCENTRATION.csv",
    index=False,
)


## 16. Timing-confidence summary

In [ ]:

timing_confidence_summary = (
    frozen.groupby(
        "_engagement_norm"
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        units_with_supported_interval=(
            "has_supported_challenge_interval",
            "sum",
        ),
        timing_confident_units=(
            "challenge_timing_confident",
            "sum",
        ),
        dynamic_timing_eligible_units=(
            "dynamic_timing_eligible",
            "sum",
        ),
        one_supported_interval_pct=(
            "supported_interval_count_timing",
            lambda s: (
                (
                    s
                    == 1
                ).mean()
                * 100.0
            )
            if len(
                s
            )
            else np.nan,
        ),
        median_dominant_interval_weight_pct=(
            "dominant_interval_weight_fraction_timing",
            lambda s: (
                s.median()
                * 100.0
            ),
        ),
        p10_dominant_interval_weight_pct=(
            "dominant_interval_weight_fraction_timing",
            lambda s: (
                s.quantile(
                    0.10
                )
                * 100.0
            ),
        ),
        median_dominant_challenge_hours=(
            "dominant_challenge_interval_hours",
            "median",
        ),
        p90_dominant_challenge_hours=(
            "dominant_challenge_interval_hours",
            lambda s: s.quantile(
                0.90
            ),
        ),
    )
    .reset_index()
)

timing_confidence_summary[
    "dynamic_timing_eligible_pct"
] = (
    timing_confidence_summary[
        "dynamic_timing_eligible_units"
    ]
    / timing_confidence_summary[
        "units"
    ]
    * 100.0
)

display(
    timing_confidence_summary
)

timing_confidence_summary.to_csv(
    OUTPUT_ROOT
    / "TIMING_CONFIDENCE_SUMMARY.csv",
    index=False,
)


## 17. Timestamp-decoder audit

In [ ]:

timestamp_decoder_audit = (
    final_timing.groupby(
        "timestamp_integer_unit"
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
        median_incident_gap_hours=(
            "incident_median_nirops_gap_hours",
            "median",
        ),
        min_incident_gap_hours=(
            "incident_median_nirops_gap_hours",
            "min",
        ),
        max_incident_gap_hours=(
            "incident_median_nirops_gap_hours",
            "max",
        ),
    )
    .reset_index()
)

display(
    timestamp_decoder_audit
)

timestamp_decoder_audit.to_csv(
    OUTPUT_ROOT
    / "TIMESTAMP_DECODER_AUDIT.csv",
    index=False,
)


## 18. Exclusion summary

In [ ]:

exclusion_summary = (
    excluded_units.groupby(
        [
            "_engagement_norm",
            "final_exclusion_reason",
        ],
        dropna=False,
    )
    .agg(
        units=(
            "unit_uid",
            "size",
        ),
        incidents=(
            "_irwin_norm",
            "nunique",
        ),
    )
    .reset_index()
)

exclusion_summary[
    "pct_of_all_corrected_units"
] = (
    exclusion_summary[
        "units"
    ]
    / len(
        unit_metrics
    )
    * 100.0
)

display(
    exclusion_summary
)

exclusion_summary.to_csv(
    OUTPUT_ROOT
    / "EXCLUSION_SUMMARY.csv",
    index=False,
)


## 19. Dataset card

In [ ]:

DATASET_CARD_PATH = (
    FROZEN_ROOT
    / "DATASET_CARD_FROZEN_V1.json"
)

dataset_card = {
    "dataset_name": (
        "Corrected local wildfire fireline effectiveness ML cohort"
    ),
    "dataset_version": "FROZEN_V1",
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook_build_id": NOTEBOOK_BUILD_ID,
    "representation": {
        "maximum_actual_unit_length_m": (
            EXPECTED_MAX_UNIT_LENGTH_M
        ),
        "maximum_fragment_endpoint_gap_m": (
            EXPECTED_FRAGMENT_GAP_M
        ),
        "within_unit_sample_spacing_m_approx": (
            EXPECTED_SAMPLE_SPACING_M
        ),
        "sample_weighting": (
            "arc-length weight = geometry part length / "
            "number of sample points on that part"
        ),
    },
    "temporal_reconstruction": {
        "nirops_raster_resolution_m": 25.0,
        "challenge_distance_tolerance_m": (
            EXPECTED_CHALLENGE_TOLERANCE_M
        ),
        "maximum_supported_nirops_interval_hours": (
            EXPECTED_MAX_NIROPS_GAP_HOURS
        ),
        "challenge_time_semantics": (
            "interval-censored between consecutive NIROPS observations"
        ),
    },
    "target_rule": {
        "burned_over": (
            "weighted_support_fraction >= 0.10"
        ),
        "held": (
            "weighted_support_fraction >= 0.10 "
            "AND weighted_conflict_fraction == 0"
        ),
        "binary_target": {
            "hold_label_1": "Held",
            "hold_label_0": "Burned Over",
        },
    },
    "dynamic_timing": {
        "primary_interval": (
            "supported NIROPS interval with maximum "
            "arc-length-supported weight"
        ),
        "tie_break": (
            "earlier interval end, then earlier interval start"
        ),
        "confidence": (
            "dominant interval supported weight / "
            "total supported event weight"
        ),
        "dynamic_timing_eligible_threshold": (
            CHALLENGE_TIMING_CONFIDENCE_MIN
        ),
    },
    "counts": {
        "corrected_units_before_final_rule": len(
            unit_metrics
        ),
        "frozen_supported_units": len(
            frozen
        ),
        "frozen_incidents": int(
            frozen[
                "_irwin_norm"
            ].nunique()
        ),
        "held_units": int(
            (
                frozen[
                    "hold_label"
                ]
                == 1
            ).sum()
        ),
        "burned_over_units": int(
            (
                frozen[
                    "hold_label"
                ]
                == 0
            ).sum()
        ),
        "dynamic_timing_eligible_units": int(
            frozen[
                "dynamic_timing_eligible"
            ].sum()
        ),
    },
    "files": {
        "frozen_cohort": str(
            FROZEN_COHORT_PATH
        ),
        "dynamic_timing_cohort": str(
            DYNAMIC_COHORT_PATH
        ),
        "excluded_units": str(
            EXCLUDED_PATH
        ),
    },
    "evaluation_requirements": [
        (
            "Never randomly split local units across train/test; "
            "all units from one incident must remain together."
        ),
        (
            "Report incident-held-out metrics and macro-by-fire metrics."
        ),
        (
            "Compare ordinary row weighting with incident-balanced "
            "and/or incident-class-balanced weighting."
        ),
        (
            "Burned Over remains concentrated in a small number of "
            "incidents; interpret aggregate row metrics accordingly."
        ),
    ],
    "scientific_cautions": [
        (
            "Fireline Held/Burned Over labels are quality-assured "
            "spatially inferred outcomes, not direct field measurements "
            "of physical line failure."
        ),
        (
            "Challenge time is interval-censored by NIROPS observation cadence."
        ),
        (
            "NIROPS coverage is operationally selected and does not provide "
            "continuous observation of every fire."
        ),
        (
            "Suppression effort and operational decisions remain latent "
            "confounders; the cohort does not establish causal suppression effects."
        ),
    ],
}

DATASET_CARD_PATH.write_text(
    json.dumps(
        dataset_card,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved:",
    DATASET_CARD_PATH
)


## 20. Final freeze report

In [ ]:

def df_to_md(
    dataframe,
    max_rows=200,
):

    if (
        dataframe is None
        or len(
            dataframe
        ) == 0
    ):

        return "_No rows._"

    try:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_markdown(
                index=False
            )
        )

    except Exception:

        return (
            dataframe
            .head(
                max_rows
            )
            .to_string(
                index=False
            )
        )


report = f"""# Frozen Corrected Wildfire Fireline ML Cohort

Notebook build: `{NOTEBOOK_BUILD_ID}`

Generated: {datetime.now(timezone.utc).isoformat()}

## Status

**FROZEN_V1**

The spatial representation and temporal target rule are frozen for downstream feature engineering.

No NIROPS rasterization was performed by 03N.

## Final target rule

Burned Over:

`weighted_support_fraction >= {SUPPORT_FRACTION_THRESHOLD:.2f}`

Held:

`weighted_support_fraction >= {SUPPORT_FRACTION_THRESHOLD:.2f} AND weighted_conflict_fraction == {HELD_CONFLICT_FRACTION_MAX:.2f}`

Binary target:

`hold_label = 1` → Held

`hold_label = 0` → Burned Over

## Challenge timing

Primary interval:

the supported NIROPS interval containing the greatest arc-length-supported evidence weight.

Dynamic timing eligibility:

`dominant_interval_weight_fraction >= {CHALLENGE_TIMING_CONFIDENCE_MIN:.2f}`

## Freeze summary

{df_to_md(freeze_summary)}

## Class summary

{df_to_md(class_summary)}

## Year × class

{df_to_md(year_class_summary)}

## Incident representation

{df_to_md(incident_representation)}

## Incident concentration

{df_to_md(incident_concentration)}

## Timing confidence

{df_to_md(timing_confidence_summary)}

## Timestamp decoder

{df_to_md(timestamp_decoder_audit)}

## Exclusions

{df_to_md(exclusion_summary)}

## Frozen files

Full target cohort:

`{FROZEN_COHORT_PATH}`

Dynamic timing-eligible cohort:

`{DYNAMIC_COHORT_PATH}`

Excluded corrected units:

`{EXCLUDED_PATH}`

Dataset card:

`{DATASET_CARD_PATH}`

## Required downstream evaluation policy

1. Split by entire incident, never by individual local unit.
2. Report row-level and macro-by-fire performance.
3. Report Held and Burned Over metrics separately.
4. Compare ordinary row weighting against incident-balanced and/or incident-class-balanced training.
5. Treat Burned Over incident concentration explicitly in interpretation.

## Dynamic feature policy

The full frozen cohort is the target dataset.

For features requiring one unique challenge interval, use only rows where:

`dynamic_timing_eligible == True`.

The dominant corrected interval should be used for later HRRR interval aggregation.

Signed trajectory caches from 03L should be reused for 04A dynamic approach features.

## Scientific wording

Use:

**quality-assured spatially inferred fireline engagement outcomes**

rather than direct physical field truth.

Use:

**interval-censored challenge timing**

for NIROPS-derived temporal interaction.

Avoid causal claims that the fireline itself caused the observed fire outcome.
"""

REPORT_PATH = (
    OUTPUT_ROOT
    / "FROZEN_COHORT_REPORT.md"
)

REPORT_PATH.write_text(
    report,
    encoding="utf-8",
)

print(
    report
)

print(
    "\nSaved:",
    REPORT_PATH
)


## 21. Output inventory

In [ ]:

print("=" * 84)
print("03N FROZEN OUTPUTS")
print("=" * 84)

for path in sorted(
    FROZEN_ROOT.rglob(
        "*"
    )
):

    if path.is_file():

        size = (
            f"{path.stat().st_size / (1024**2):,.2f} MB"
            if (
                path.stat().st_size
                >= 1024**2
            )
            else f"{path.stat().st_size / 1024:,.1f} KB"
        )

        print(
            f"{str(path.relative_to(FROZEN_ROOT)):<72}"
            f"{size:>15}"
        )

print()

for path in sorted(
    OUTPUT_ROOT.rglob(
        "*"
    )
):

    if path.is_file():

        size = (
            f"{path.stat().st_size / (1024**2):,.2f} MB"
            if (
                path.stat().st_size
                >= 1024**2
            )
            else f"{path.stat().st_size / 1024:,.1f} KB"
        )

        print(
            f"{str(path.relative_to(OUTPUT_ROOT)):<72}"
            f"{size:>15}"
        )

print()
print(
    "DONE:",
    NOTEBOOK_BUILD_ID
)


In [ ]:
# Numerical-boundary audit for the final 10% rule

threshold = 0.10
tol = 1e-12

near_boundary = unit_metrics[
    (unit_metrics["weighted_support_fraction"] < threshold)
    & (
        unit_metrics["weighted_support_fraction"]
        >= threshold - tol
    )
].copy()

print("Rows just below 0.10 but within tolerance:")
print(f"{len(near_boundary):,}")

print("\nBy class:")
display(
    near_boundary[
        "_engagement_norm"
    ].value_counts()
)

print("\nSupport fraction range:")
display(
    near_boundary[
        "weighted_support_fraction"
    ].describe()
)

near_boundary["distance_from_0.10"] = (
    threshold
    - near_boundary["weighted_support_fraction"]
)

print("\nLargest numerical deviation below 0.10:")
print(
    near_boundary[
        "distance_from_0.10"
    ].max()
)

# Also verify that the zero-conflict tolerance is not admitting
# genuinely conflicting Held units.
tiny_positive_conflict = unit_metrics[
    (unit_metrics["_engagement_norm"] == "Held")
    & (unit_metrics["weighted_conflict_fraction"] > 0)
    & (unit_metrics["weighted_conflict_fraction"] <= tol)
]

print("\nHeld rows with tiny positive conflict <= 1e-12:")
print(f"{len(tiny_positive_conflict):,}")

In [ ]:
held_boundary = near_boundary[
    near_boundary["_engagement_norm"] == "Held"
]

print(
    "Held boundary with zero conflict:",
    (
        held_boundary["weighted_conflict_fraction"] == 0
    ).sum()
)

print(
    "Held boundary with real positive conflict:",
    (
        held_boundary["weighted_conflict_fraction"] > 0
    ).sum()
)